# Chủ đề: Nghiên cứu xây dựng mô hình học đa tác vụ kết hợp xử lý mất cân bằng lớp cho phân tích phản hồi sinh viên/UIT-VSFC
## Người thực hiện: Nguyễn Thị Thu Trang
## MSSV: 22028254

# Cài đặt import các thư viện

In [1]:
!pip install -q -U google-genai
!pip install gdown

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 818.2/818.2 kB 15.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.1/246.1 kB 13.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.47.0, but you have google-auth 2.53.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
google-cloud-aiplatform 1.138.0 requires google-genai<2.0.0,>=1.59.0; python_versio

# Tải dữ liệu UIT-VSFC từ Google Drive

In [3]:
import gdown
import pandas as pd
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
import math
import random
import os
# URL gốc trên Google Drive 
URLS = {
    "train": {
        "sentences": "https://drive.google.com/uc?id=1nzak5OkrheRV1ltOGCXkT671bmjODLhP",
        "sentiments": "https://drive.google.com/uc?id=1ye-gOZIBqXdKOoi_YxvpT6FeRNmViPPv",
        "topics": "https://drive.google.com/uc?id=14MuDtwMnNOcr4z_8KdpxprjbwaQ7lJ_C",
    },
    "validation": {
        "sentences": "https://drive.google.com/uc?id=1sMJSR3oRfPc3fe1gK-V3W5F24tov_517",
        "sentiments": "https://drive.google.com/uc?id=1GiY1AOp41dLXIIkgES4422AuDwmbUseL",
        "topics": "https://drive.google.com/uc?id=1DwLgDEaFWQe8mOd7EpF-xqMEbDLfdT-W",
    },
    "test": {
        "sentences": "https://drive.google.com/uc?id=1aNMOeZZbNwSRkjyCWAGtNCMa3YrshR-n",
        "sentiments": "https://drive.google.com/uc?id=1vkQS5gI0is4ACU58-AbWusnemw7KZNfO",
        "topics": "https://drive.google.com/uc?id=1_ArMpDguVsbUGl-xSMkTF_p5KpZrmpSB",
    },
}


# Tải và đọc từng split 
def load_split(name, urls):
    files = {}
    for k, url in urls.items():
        output = f"{name}_{k}.txt"
        gdown.download(url, output, quiet=True)
        files[k] = output

    # Đọc 3 file và gộp lại
    with open(files["sentences"], encoding="utf-8") as f:
        sentences = [line.strip() for line in f]
    with open(files["sentiments"], encoding="utf-8") as f:
        sentiments = [int(line.strip()) for line in f]
    with open(files["topics"], encoding="utf-8") as f:
        topics = [int(line.strip()) for line in f]

    return pd.DataFrame({
        "sentence": sentences,
        "sentiment": sentiments,
        "topic": topics
    })

# Tải từng tập
train_df = load_split("train", URLS["train"])

# GDA train cũ

In [ ]:
import pandas as pd
import time
import json
import math
import os
from tqdm import tqdm
from google import genai
from google.genai import types

# 1. CẤU HÌNH API VÀ SAFETY SETTINGS
API_KEY = "API_CUA_BAN" 

client = genai.Client(api_key=API_KEY)
MODEL_NAME = 'gemini-2.5-flash'

safe_config = [
    types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HARASSMENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
    types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH, threshold=types.HarmBlockThreshold.BLOCK_NONE),
    types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
    types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
]


# 2. TỪ ĐIỂN MAP NHÃN & MÔ TẢ (MTL)
# Sentiment (0: Negative, 1: Neutral, 2: Positive)
# Topic (0: Lecturer, 1: Curriculum, 2: Facility, 3: Others)

SENTIMENT_MAP = {0: "Negative", 1: "Neutral", 2: "Positive"}
TOPIC_MAP = {0: "Lecturer", 1: "Curriculum", 2: "Facility", 3: "Others"}

SENTIMENT_DESC = {
    "Positive": "tích cực, hài lòng, khen ngợi",
    "Negative": "tiêu cực, phàn nàn, góp ý cải thiện",
    "Neutral":  "trung tính, chỉ mô tả khách quan, không khen không chê",
}

TOPIC_DESC = {
    "Lecturer":    "giảng viên: cách dạy, thái độ, chấm điểm",
    "Facility":    "cơ sở vật chất: phòng học, wifi, máy chiếu, điều hòa, phòng máy",
    "Others":      "vấn đề khác: thủ tục hành chính, học bổng, lịch thi, quy định",
    "Curriculum":  "chương trình học: môn học, tín chỉ, giáo trình, bài tập, kiểm tra",
}


# 3. CHUẨN BỊ DỮ LIỆU & LOGIC HỆ SỐ NHÂN
print("[1] Đang đọc dữ liệu từ Input...")
# input_path = "/kaggle/input/datasets/gnart20044/train-val-relabled/train_final.csv"
backup_file = "/kaggle/working/old_train_llm_batch_backup_gda.csv"
final_file = "/kaggle/working/old_train_llm_final_gda.csv"

# df = pd.read_csv(input_path)
df = train_df
if 'is_augmented' not in df.columns:
    df['is_augmented'] = False
if 'original_id' not in df.columns:
    df['original_id'] = -1 # Để track xem câu sinh ra thuộc về câu gốc nào

# Hàm xác định số lượng câu cần sinh (Multiplier)
def get_n_variants(row):
    n = 0
    sent_label = SENTIMENT_MAP.get(row['sentiment'], "")
    top_label = TOPIC_MAP.get(row['topic'], "")
    
    # Logic theo IR 
    if sent_label == "Neutral":    n = max(n, 2)
    if top_label == "Facility":    n = max(n, 3)
    if top_label == "Others":      n = max(n, 2)
    if top_label == "Curriculum":  n = max(n, 1)
    
    return n, sent_label, top_label

# Lọc ra danh sách các câu CẦN sinh thêm
tasks = []
for idx, row in df.iterrows():
    n_vars, sent_str, top_str = get_n_variants(row)
    if n_vars > 0:
        tasks.append({
            'id': idx, 
            'sentence': row['sentence'],
            'n_variants': n_vars,
            'sentiment_desc': SENTIMENT_DESC.get(sent_str, ""),
            'topic_desc': TOPIC_DESC.get(top_str, "")
        })

tasks_df = pd.DataFrame(tasks)
print(f"-> Tổng số câu gốc cần đem đi Augment: {len(tasks_df)}")

# LOGIC AUTO-RESUME (CHẠY TIẾP KHI ĐỔI KEY) 
new_rows = []
if os.path.exists(backup_file):
    print("-> [PHÁT HIỆN FILE BACKUP] Đang lọc các câu đã hoàn thành...")
    df_backup = pd.read_csv(backup_file)
    
    augmented_df = df_backup[df_backup['is_augmented'] == True]
    if len(augmented_df) > 0:
        # Lấy danh sách ID gốc đã được xử lý thành công
        processed_ids = augmented_df['original_id'].unique()
        
        # Loại bỏ các ID đã xử lý khỏi danh sách cần chạy
        tasks_df = tasks_df[~tasks_df['id'].isin(processed_ids)]
        
        # Nạp lại dữ liệu cũ vào bộ nhớ
        new_rows = augmented_df.to_dict('records')
        print(f"-> Đã bỏ qua {len(processed_ids)} câu gốc. Đã nạp {len(new_rows)} câu biến thể vào bộ nhớ.")

print(f"-> Số lượng câu gốc CÒN LẠI cần gọi API: {len(tasks_df)}")


# 4. CHẠY VÒNG LẶP GỌI API (BATCHING)

BATCH_SIZE = 20 
num_batches = math.ceil(len(tasks_df) / BATCH_SIZE)

SYSTEM_PROMPT = """Bạn là một chuyên gia dữ liệu đóng vai sinh viên đại học Việt Nam.
Tôi sẽ cung cấp một mảng JSON chứa các câu phản hồi. Mỗi câu có yêu cầu về số lượng (n), thái độ (sentiment) và chủ đề (topic).

NHIỆM VỤ CỦA BẠN: Viết lại TỪNG CÂU thành đúng `n` câu KHÁC NHAU.
- Tuân thủ tuyệt đối THÁI ĐỘ và CHỦ ĐỀ được yêu cầu.
- Văn phong: ngắn gọn, tự nhiên như sinh viên viết (có thể dùng viết tắt phổ biến như sv, gv, ktx, wifi...).
- Tránh lặp từ ngữ, đa dạng cách diễn đạt.

ĐẦU RA BẮT BUỘC LÀ JSON CÓ CẤU TRÚC SAU (KHÔNG DÙNG MARKDOWN KHÁC):
[
    {
        "id": 123,
        "variants": ["câu 1", "câu 2", ...]
    }
]"""

if num_batches > 0:
    print(f"\n[2] Bắt đầu sinh dữ liệu ({num_batches} lô)...")

    for i in tqdm(range(num_batches), desc="Processing Batches"):
        start_idx = i * BATCH_SIZE
        end_idx = min((i + 1) * BATCH_SIZE, len(tasks_df))
        batch_tasks = tasks_df.iloc[start_idx:end_idx].to_dict('records')
        
        # Chuẩn bị input dạng JSON cho Gemini dễ hiểu cấu trúc phức tạp
        batch_input_json = json.dumps(batch_tasks, ensure_ascii=False, indent=2)
        prompt = f"{SYSTEM_PROMPT}\n\nDANH SÁCH CẦN XỬ LÝ:\n{batch_input_json}"
        
        attempt = 0
        while True:
            try:
                attempt += 1
                response = client.models.generate_content(
                    model=MODEL_NAME,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        response_mime_type="application/json",
                        temperature=0.85, # Tăng nhẹ temp để sinh câu đa dạng hơn
                        safety_settings=safe_config
                    )
                )
                
                if not response.text:
                    time.sleep(3)
                    continue
                    
                results = json.loads(response.text)
                
                # Ghi nhận kết quả
                for res in results:
                    original_idx = int(res['id'])
                    variants = res.get('variants', [])
                    
                    if original_idx in df.index:
                        original_row = df.loc[original_idx]
                        
                        # Chỉ lấy đúng số lượng n (tránh LLM sinh dư)
                        target_n = tasks_df[tasks_df['id'] == original_idx]['n_variants'].values[0]
                        variants = [v for v in variants if v and isinstance(v, str)][:target_n]
                        
                        for v in variants:
                            new_row = original_row.copy()
                            new_row['sentence'] = v.strip()
                            new_row['is_augmented'] = True
                            new_row['original_id'] = original_idx # Lưu ID gốc để Tracking Resume
                            if 'sentence_seg' in new_row:
                                new_row['sentence_seg'] = "" 
                            new_rows.append(new_row)
                
                # AUTO-SAVE CỨ SAU MỖI BATCH
                if len(new_rows) > 0:
                    temp_df = pd.DataFrame(new_rows)
                    pd.concat([df, temp_df], ignore_index=True).to_csv(backup_file, index=False)
                
                time.sleep(5) # Delay mượt mà để tránh rate limit
                break 
                
            except Exception as e:
                error_msg = str(e).lower()
                if "429" in error_msg or "quota" in error_msg or "exhausted" in error_msg:
                    print(f"\n[QUOTA EXHAUSTED] Đã hết lượt API hoặc Rate Limit (Lần {attempt}).")
                    print("-> BẠN CÓ THỂ DỪNG CELL NÀY (STOP), THAY API KEY MỚI Ở TRÊN, VÀ CHẠY LẠI TỪ ĐẦU.")
                    print("-> Dữ liệu đã được lưu an toàn trong file backup!")
                    time.sleep(20) # Đợi lâu hơn nếu bị rate limit
                elif "503" in error_msg or "500" in error_msg:
                    print(f"\nLỗi Server Google. Đợi 10s...")
                    time.sleep(10) 
                elif isinstance(e, json.JSONDecodeError):
                    print(f"\nLỗi JSON lô {i+1}. Gen lại...")
                    time.sleep(2)
                else:
                    print(f"\n[!] Lỗi bất ngờ: {e}. Bỏ qua lô {i+1}.")
                    break 
else:
    print("\n[2] Dữ liệu đã chạy xong 100% từ trước! Bỏ qua vòng lặp API.")


# 5. HỢP NHẤT VÀ LƯU FILE CUỐI CÙNG

if len(new_rows) > 0:
    print("\n[3] Đang lưu file kết quả chính thức...")
    df_llm_augmented = pd.DataFrame(new_rows)
    
    # Gỡ bỏ cột original_id vì không cần cho train model nữa
    if 'original_id' in df.columns:
        df = df.drop(columns=['original_id'])
    if 'original_id' in df_llm_augmented.columns:
        df_llm_augmented = df_llm_augmented.drop(columns=['original_id'])
        
    train_llm_df = pd.concat([df, df_llm_augmented], ignore_index=True)
    
    # Shuffle toàn bộ data
    train_llm_df = train_llm_df.sample(frac=1, random_state=42).reset_index(drop=True)
    train_llm_df.to_csv(final_file, index=False)
    
    print(f"-> HOÀN TẤT! File SẴN SÀNG ĐỂ TRAIN lưu tại: {final_file}")
    print(f"-> Tổng số dòng hiện tại: {len(train_llm_df)} (Đã mồi thêm {len(df_llm_augmented)} mẫu GDA).")
else:
    print("\n[3] Không có dữ liệu mới nào được sinh ra.")

[1] Đang đọc dữ liệu từ Input...
-> Tổng số câu gốc cần đem đi Augment: 3446
-> [PHÁT HIỆN FILE BACKUP] Đang lọc các câu đã hoàn thành...
-> Đã bỏ qua 3200 câu gốc. Đã nạp 4912 câu biến thể vào bộ nhớ.
-> Số lượng câu gốc CÒN LẠI cần gọi API: 246

[2] Bắt đầu sinh dữ liệu (13 lô)...


Processing Batches: 100%|██████████| 13/13 [06:11<00:00, 28.61s/it]


[3] Đang lưu file kết quả chính thức...
-> HOÀN TẤT! File SẴN SÀNG ĐỂ TRAIN lưu tại: /kaggle/working/old_train_llm_final_gda.csv
-> Tổng số dòng hiện tại: 16715 (Đã mồi thêm 5289 mẫu GDA).


# GDA train final

In [ ]:
# !pip install -q -U google-genai pandas tqdm
import pandas as pd
import time
import json
import math
import os
from tqdm import tqdm
from google import genai
from google.genai import types

# 1. CẤU HÌNH API VÀ SAFETY SETTINGS
API_KEY = "API_CUA_BAN" 

client = genai.Client(api_key=API_KEY)
MODEL_NAME = 'gemini-2.5-flash'

safe_config = [
    types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HARASSMENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
    types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_HATE_SPEECH, threshold=types.HarmBlockThreshold.BLOCK_NONE),
    types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
    types.SafetySetting(category=types.HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT, threshold=types.HarmBlockThreshold.BLOCK_NONE),
]

# 2. TỪ ĐIỂN MAP NHÃN & MÔ TẢ (MTL)
# Sentiment (0: Negative, 1: Neutral, 2: Positive)
# Topic (0: Lecturer, 1: Curriculum, 2: Facility, 3: Others)

SENTIMENT_MAP = {0: "Negative", 1: "Neutral", 2: "Positive"}
TOPIC_MAP = {0: "Lecturer", 1: "Curriculum", 2: "Facility", 3: "Others"}

SENTIMENT_DESC = {
    "Positive": "tích cực, hài lòng, khen ngợi",
    "Negative": "tiêu cực, phàn nàn, góp ý cải thiện",
    "Neutral":  "trung tính, chỉ mô tả khách quan, không khen không chê",
}

TOPIC_DESC = {
    "Lecturer":    "giảng viên: cách dạy, thái độ, chấm điểm",
    "Facility":    "cơ sở vật chất: phòng học, wifi, máy chiếu, điều hòa, phòng máy",
    "Others":      "vấn đề khác: thủ tục hành chính, học bổng, lịch thi, quy định",
    "Curriculum":  "chương trình học: môn học, tín chỉ, giáo trình, bài tập, kiểm tra",
}


# 3. CHUẨN BỊ DỮ LIỆU & LOGIC HỆ SỐ NHÂN

print("[1] Đang đọc dữ liệu từ Input...")
input_path = "/kaggle/input/datasets/gnart20044/train-val-relabled/train_final.csv"
backup_file = "/kaggle/working/train_llm_batch_backup_gda.csv"
final_file = "/kaggle/working/train_llm_final_gda.csv"

df = pd.read_csv(input_path)
if 'is_augmented' not in df.columns:
    df['is_augmented'] = False
if 'original_id' not in df.columns:
    df['original_id'] = -1 # Để track xem câu sinh ra thuộc về câu gốc nào

# Hàm xác định số lượng câu cần sinh (Multiplier)
def get_n_variants(row):
    n = 0
    sent_label = SENTIMENT_MAP.get(row['sentiment'], "")
    top_label = TOPIC_MAP.get(row['topic'], "")
    
    # Logic theo IR bạn cung cấp
    if sent_label == "Neutral":    n = max(n, 2)
    if top_label == "Facility":    n = max(n, 3)
    if top_label == "Others":      n = max(n, 2)
    if top_label == "Curriculum":  n = max(n, 1)
    
    return n, sent_label, top_label

# Lọc ra danh sách các câu CẦN sinh thêm
tasks = []
for idx, row in df.iterrows():
    n_vars, sent_str, top_str = get_n_variants(row)
    if n_vars > 0:
        tasks.append({
            'id': idx, 
            'sentence': row['sentence'],
            'n_variants': n_vars,
            'sentiment_desc': SENTIMENT_DESC.get(sent_str, ""),
            'topic_desc': TOPIC_DESC.get(top_str, "")
        })

tasks_df = pd.DataFrame(tasks)
print(f"-> Tổng số câu gốc cần đem đi Augment: {len(tasks_df)}")

# --- LOGIC AUTO-RESUME (CHẠY TIẾP KHI ĐỔI KEY) ---
new_rows = []
if os.path.exists(backup_file):
    print("-> [PHÁT HIỆN FILE BACKUP] Đang lọc các câu đã hoàn thành...")
    df_backup = pd.read_csv(backup_file)
    
    augmented_df = df_backup[df_backup['is_augmented'] == True]
    if len(augmented_df) > 0:
        # Lấy danh sách ID gốc đã được xử lý thành công
        processed_ids = augmented_df['original_id'].unique()
        
        # Loại bỏ các ID đã xử lý khỏi danh sách cần chạy
        tasks_df = tasks_df[~tasks_df['id'].isin(processed_ids)]
        
        # Nạp lại dữ liệu cũ vào bộ nhớ
        new_rows = augmented_df.to_dict('records')
        print(f"-> Đã bỏ qua {len(processed_ids)} câu gốc. Đã nạp {len(new_rows)} câu biến thể vào bộ nhớ.")

print(f"-> Số lượng câu gốc CÒN LẠI cần gọi API: {len(tasks_df)}")


# 4. CHẠY VÒNG LẶP GỌI API (BATCHING)
BATCH_SIZE = 20 
num_batches = math.ceil(len(tasks_df) / BATCH_SIZE)

SYSTEM_PROMPT = """Bạn là một chuyên gia dữ liệu đóng vai sinh viên đại học Việt Nam.
Tôi sẽ cung cấp một mảng JSON chứa các câu phản hồi. Mỗi câu có yêu cầu về số lượng (n), thái độ (sentiment) và chủ đề (topic).

NHIỆM VỤ CỦA BẠN: Viết lại TỪNG CÂU thành đúng `n` câu KHÁC NHAU.
- Tuân thủ tuyệt đối THÁI ĐỘ và CHỦ ĐỀ được yêu cầu.
- Văn phong: ngắn gọn, tự nhiên như sinh viên viết (có thể dùng viết tắt phổ biến như sv, gv, ktx, wifi...).
- Tránh lặp từ ngữ, đa dạng cách diễn đạt.

ĐẦU RA BẮT BUỘC LÀ JSON CÓ CẤU TRÚC SAU (KHÔNG DÙNG MARKDOWN KHÁC):
[
    {
        "id": 123,
        "variants": ["câu 1", "câu 2", ...]
    }
]"""

if num_batches > 0:
    print(f"\n[2] Bắt đầu sinh dữ liệu ({num_batches} lô)...")

    for i in tqdm(range(num_batches), desc="Processing Batches"):
        start_idx = i * BATCH_SIZE
        end_idx = min((i + 1) * BATCH_SIZE, len(tasks_df))
        batch_tasks = tasks_df.iloc[start_idx:end_idx].to_dict('records')
        
        # Chuẩn bị input dạng JSON cho Gemini dễ hiểu cấu trúc phức tạp
        batch_input_json = json.dumps(batch_tasks, ensure_ascii=False, indent=2)
        prompt = f"{SYSTEM_PROMPT}\n\nDANH SÁCH CẦN XỬ LÝ:\n{batch_input_json}"
        
        attempt = 0
        while True:
            try:
                attempt += 1
                response = client.models.generate_content(
                    model=MODEL_NAME,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        response_mime_type="application/json",
                        temperature=0.85, # Tăng nhẹ temp để sinh câu đa dạng hơn
                        safety_settings=safe_config
                    )
                )
                
                if not response.text:
                    time.sleep(3)
                    continue
                    
                results = json.loads(response.text)
                
                # Ghi nhận kết quả
                for res in results:
                    original_idx = int(res['id'])
                    variants = res.get('variants', [])
                    
                    if original_idx in df.index:
                        original_row = df.loc[original_idx]
                        
                        # Chỉ lấy đúng số lượng n (tránh LLM sinh dư)
                        target_n = tasks_df[tasks_df['id'] == original_idx]['n_variants'].values[0]
                        variants = [v for v in variants if v and isinstance(v, str)][:target_n]
                        
                        for v in variants:
                            new_row = original_row.copy()
                            new_row['sentence'] = v.strip()
                            new_row['is_augmented'] = True
                            new_row['original_id'] = original_idx # Lưu ID gốc để Tracking Resume
                            if 'sentence_seg' in new_row:
                                new_row['sentence_seg'] = "" 
                            new_rows.append(new_row)
                
                # AUTO-SAVE CỨ SAU MỖI BATCH
                if len(new_rows) > 0:
                    temp_df = pd.DataFrame(new_rows)
                    pd.concat([df, temp_df], ignore_index=True).to_csv(backup_file, index=False)
                
                time.sleep(5) # Delay mượt mà để tránh rate limit
                break 
                
            except Exception as e:
                error_msg = str(e).lower()
                if "429" in error_msg or "quota" in error_msg or "exhausted" in error_msg:
                    print(f"\n[QUOTA EXHAUSTED] Đã hết lượt API hoặc Rate Limit (Lần {attempt}).")
                    print("-> BẠN CÓ THỂ DỪNG CELL NÀY (STOP), THAY API KEY MỚI Ở TRÊN, VÀ CHẠY LẠI TỪ ĐẦU.")
                    print("-> Dữ liệu đã được lưu an toàn trong file backup!")
                    time.sleep(20) # Đợi lâu hơn nếu bị rate limit
                elif "503" in error_msg or "500" in error_msg:
                    print(f"\nLỗi Server Google. Đợi 10s...")
                    time.sleep(10) 
                elif isinstance(e, json.JSONDecodeError):
                    print(f"\nLỗi JSON lô {i+1}. Gen lại...")
                    time.sleep(2)
                else:
                    print(f"\n[!] Lỗi bất ngờ: {e}. Bỏ qua lô {i+1}.")
                    break 
else:
    print("\n[2] Dữ liệu đã chạy xong 100% từ trước! Bỏ qua vòng lặp API.")


# 5. HỢP NHẤT VÀ LƯU FILE CUỐI CÙNG
if len(new_rows) > 0:
    print("\n[3] Đang lưu file kết quả chính thức...")
    df_llm_augmented = pd.DataFrame(new_rows)
    
    # Gỡ bỏ cột original_id vì không cần cho train model nữa
    if 'original_id' in df.columns:
        df = df.drop(columns=['original_id'])
    if 'original_id' in df_llm_augmented.columns:
        df_llm_augmented = df_llm_augmented.drop(columns=['original_id'])
        
    train_llm_df = pd.concat([df, df_llm_augmented], ignore_index=True)
    
    # Shuffle toàn bộ data
    train_llm_df = train_llm_df.sample(frac=1, random_state=42).reset_index(drop=True)
    train_llm_df.to_csv(final_file, index=False)
    
    print(f"-> HOÀN TẤT! File SẴN SÀNG ĐỂ TRAIN lưu tại: {final_file}")
    print(f"-> Tổng số dòng hiện tại: {len(train_llm_df)} (Đã mồi thêm {len(df_llm_augmented)} mẫu GDA).")
else:
    print("\n[3] Không có dữ liệu mới nào được sinh ra.")

[1] Đang đọc dữ liệu từ Input...
-> Tổng số câu gốc cần đem đi Augment: 3385
-> [PHÁT HIỆN FILE BACKUP] Đang lọc các câu đã hoàn thành...
-> Đã bỏ qua 3140 câu gốc. Đã nạp 4772 câu biến thể vào bộ nhớ.
-> Số lượng câu gốc CÒN LẠI cần gọi API: 245

[2] Bắt đầu sinh dữ liệu (13 lô)...


Processing Batches:   8%|▊         | 1/13 [00:24<04:51, 24.26s/it]


Lỗi Server Google. Đợi 10s...


Processing Batches:  15%|█▌        | 2/13 [01:03<06:03, 33.04s/it]


Lỗi Server Google. Đợi 10s...


...


Lỗi Server Google. Đợi 10s...

Lỗi Server Google. Đợi 10s...


Processing Batches:  46%|████▌     | 6/13 [04:46<06:45, 57.92s/it]


Lỗi Server Google. Đợi 10s...


Processing Batches: 100%|██████████| 13/13 [08:32<00:00, 39.43s/it]


[3] Đang lưu file kết quả chính thức...
-> HOÀN TẤT! File SẴN SÀNG ĐỂ TRAIN lưu tại: /kaggle/working/train_llm_final_gda.csv
-> Tổng số dòng hiện tại: 16568 (Đã mồi thêm 5142 mẫu GDA).
